---
title: "Chapter 13: Performance"
short_title: "13. Performance"
---

## 1. Measurement method

- **Environment:** Google Colab, **CPU-only** runtime, with the same code commit (tag `m2-final`) for all five ranges
  (`src/notebooks/m2_benchmark_colab.ipynb`).
- **Execution:** the Milestone 1 data of each range is restored to the runtime's local disk (`/content`; nothing is
  parsed directly on Google Drive); the ranges are parsed **one after another**, each with one
  `benchmark_m2.py --force` run using 2 processes (`--workers 2`). The exported results
  record no re-download of `.bbl` files ([Chapter 11](./11_references.ipynb)).
- **Merging:** once all five ranges are done, `pipeline_m2.py --export-only` merges the three Parquet tables of the
  group; this step is measured too.
- **Wall time** is the real time from the start to the end of the command. **Peak RAM** is the largest RSS sample of
  the whole process tree (sampled every 0.5 s), so it is a sampled value, not an absolute peak. **Output size** is the
  size of the final Parquet files.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils_m2 import (COLORS, KIND_LABELS, PROBLEM_LABELS, group, human_bytes, human_seconds, load,
                             members, meta, metric, metric_table, num, ordered, pct, percent, ready, secs,
                             show, table)
READY = ready()

In [ ]:
if READY:
    info = meta().get("ranges") or {}
    env = pd.DataFrame([{"Range": k, "CPU": r.get("cpu_model"), "Logical CPUs": r.get("cpu_logical"),
                         "Processes": r.get("workers"), "Commit": (r.get("commit") or "")[:10], "run_id": r.get("run_id"),
                         "Finished (UTC)": r.get("finished_utc")} for k, r in info.items()])
    table(ordered(env, "Range"), "Table 13.1. Environment of each parsing run")

## 2. Results per range

In [ ]:
if READY:
    m, g = members(), group()
    runs = load("benchmark_runs")
    parse = runs[runs["stage"] == "parse"].set_index("member")
    rows = []
    for _, r in m.iterrows():
        papers, wall = float(r["papers"]), float(r["wall_seconds"])
        run = parse.loc[r["member"]] if r["member"] in parse.index else None
        rows.append({"Range": r["member"], "Papers": num(papers), "Wall time": human_seconds(wall),
                     "Mean per paper": secs(wall / papers) if papers else "–",
                     "Peak RAM": human_bytes(float(r["ram_peak_bytes"])),
                     "Parquet": human_bytes(pd.to_numeric(run["parquet_final_bytes"], errors="coerce")) if run is not None else "–"})
    export = runs[runs["stage"] == "export"]
    if not export.empty:
        e = export.iloc[-1]
        rows.append({"Range": "Table merge (--export-only)", "Papers": num(float(g["papers"])),
                     "Wall time": human_seconds(float(e["wall_seconds"])), "Mean per paper": "–",
                     "Peak RAM": human_bytes(pd.to_numeric(e["ram_sampled_peak_bytes"], errors="coerce")), "Parquet": "–"})
    gw, gp = float(g["wall_seconds"]), float(g["papers"])
    rows.append({"Range": "Group (end-to-end)", "Papers": num(gp), "Wall time": human_seconds(gw),
                 "Mean per paper": secs(gw / gp) if gp else "–", "Peak RAM": human_bytes(float(g["ram_peak_bytes"])),
                 "Parquet": human_bytes(pd.to_numeric(g["parquet_final_bytes"], errors="coerce"))})
    table(pd.DataFrame(rows), "Table 13.2. Wall time, mean time per paper, peak RAM and output size")
    fig = px.bar(m.assign(minutes=pd.to_numeric(m["wall_seconds"]) / 60), x="member", y="minutes", text_auto=".1f",
                 color_discrete_sequence=COLORS, title="Parsing wall time per range",
                 labels={"member": "Range", "minutes": "Minutes"})
    show(fig)

The **group** row sums the wall times of the five parsing runs and of the merge (the end-to-end time when run one
after another on one machine), takes the largest RAM of all runs, and measures the size of the **three merged tables**.
"Mean per paper" is the wall time divided by the number of papers, i.e. the throughput with 2 processes in parallel.

## 3. Time to process one paper

The time one process spends parsing one paper (reading every version, building the tree and the references, writing
`parsed/`), recorded in `parse_log.csv`. This is process time, not wall time:

In [ ]:
if READY:
    p = load("papers")
    p["seconds"] = pd.to_numeric(p["seconds"], errors="coerce")
    s = p[p["seconds"] > 0]
    fig = px.box(ordered(s), x="member", y="seconds", log_y=True, color_discrete_sequence=COLORS,
                 title="Parsing time of one paper per range", labels={"member": "Range", "seconds": "Seconds (log scale)"})
    show(fig)
    d = s["seconds"]
    table(pd.DataFrame({"Statistic": ["Papers", "Mean", "Median", "95th percentile", "99th percentile", "Maximum"],
                        "Value": [num(len(d)), secs(d.mean()), secs(d.median()),
                                  secs(d.quantile(0.95)), secs(d.quantile(0.99)), secs(d.max())]}),
          "Table 13.3. Parsing time of one paper, whole group")

## 4. RAM over time

In [ ]:
if READY:
    sm = load("performance_samples")
    if sm.empty:
        print("No RAM samples in the exported data.")
    else:
        for c in ["elapsed_seconds", "rss_tree_bytes"]:
            sm[c] = pd.to_numeric(sm[c], errors="coerce")
        sm["Minutes"] = sm["elapsed_seconds"] / 60
        sm["RAM (MB)"] = sm["rss_tree_bytes"] / 2**20
        sm["Run"] = sm["member"].replace({"export": "Table merge"})
        fig = px.line(sm, x="Minutes", y="RAM (MB)", color="Run", color_discrete_sequence=COLORS,
                      title="RAM of the process tree over time, one line per run",
                      labels={"Minutes": "Time since the start of the run (minutes)"})
        show(fig)

RAM fluctuates during a run and is higher on average in the last quarter of each run than in the first, with peaks
that differ between ranges (Table 13.2). Each process keeps one paper in memory and writes its result to `parsed/`
as soon as it is done, and the Parquet tables are written in parts. RSS adds up the child processes, so
shared memory pages may be counted more than once.